# 01 Validate and Profile Input Dataset

This notebook validates the prepared YOLO input dataset for factory sign detection and creates the minimal pre-split profile needed by Notebook 02.


## Purpose

Notebook 01 checks image-label pairing, YOLO label format, readable images, class IDs, and bounding-box validity. It also counts no-sign images and class presence so Notebook 02 can build a balanced train/val/test split.

It does not split data, augment images, train models, create `reports/eda`, or modify files under `data/input`.


## 1. Setup

Find the `sign-detection` root, add it to `sys.path`, and import the small helper functions used by this notebook.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display


# Notebook execution can start from the repository root or from notebooks/.
# This helper walks upward until it finds the sign-detection project layout.
def find_project_root(start: Path) -> Path:
    """Return the sign-detection project root from a notebook working directory."""
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "class_names.yaml").exists() and (
            candidate / "src"
        ).exists():
            return candidate
        sign_child = candidate / "sign-detection"
        if (sign_child / "configs" / "class_names.yaml").exists() and (
            sign_child / "src"
        ).exists():
            return sign_child
    raise RuntimeError("Could not find sign-detection project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# Add the project root so imports like src.dataset.validate_yolo_dataset work in Jupyter.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Keep the notebook as orchestration only; reusable validation/profile logic lives in src/.
from src.dataset.validate_yolo_dataset import validate_yolo_dataset
from src.analysis.dataset_eda import (
    build_bbox_records,
    build_class_distribution,
    build_image_level_profile,
    build_input_summary,
)
from src.analysis.plot_utils import plot_class_distribution

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Github\smart-factory-safety-monitoring-system\sign-detection


## 2. Load Configuration

Class names and dataset paths come from config files. Paths are resolved relative to `sign-detection`, so the notebook remains portable.


In [2]:
# Load the fixed class mapping and dataset paths from config.
# This keeps the notebook portable and avoids absolute paths.
class_config_path = PROJECT_ROOT / "configs" / "class_names.yaml"
dataset_config_path = PROJECT_ROOT / "configs" / "dataset_config.yaml"

with class_config_path.open("r", encoding="utf-8") as file:
    class_config = yaml.safe_load(file)
with dataset_config_path.open("r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

# Convert class IDs to int because YAML may load mapping keys as strings.
class_names = {int(class_id): name for class_id, name in class_config["names"].items()}
class_ids = set(class_names.keys())
image_extensions = dataset_config["image_extensions"]

# Input folders are read-only for this notebook.
images_dir = PROJECT_ROOT / dataset_config["paths"]["input_images"]
labels_dir = PROJECT_ROOT / dataset_config["paths"]["input_labels"]

# Notebook 01 writes only validation and lightweight profile reports.
validation_dir = PROJECT_ROOT / dataset_config["paths"].get(
    "reports_validation", "reports/validation"
)
profile_dir = PROJECT_ROOT / "reports" / "profile"

# Report folders are generated artifacts and can be created safely.
validation_dir.mkdir(parents=True, exist_ok=True)
profile_dir.mkdir(parents=True, exist_ok=True)

print("Images:", images_dir)
print("Labels:", labels_dir)
print("Supported image extensions:", image_extensions)
print("Classes:", class_names)


Images: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\input\images
Labels: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\input\labels
Supported image extensions: ['.jpg', '.jpeg', '.png']
Classes: {0: 'M014_Helmet', 1: 'M015_Vest', 2: 'P004_NoThoroughfare', 3: 'W011_Slippery'}


## 3. Validate YOLO Input Data

Every discovered image or unmatched label is represented in `validation_report.csv`. Empty label files are valid no-sign images and are counted with `is_no_sign = True`.


In [3]:
# Validate every discovered image-label pair without modifying input files.
# Empty .txt labels are valid no-sign images and are counted separately.
validation_df = validate_yolo_dataset(
    images_dir=images_dir,
    labels_dir=labels_dir,
    class_ids=class_ids,
    image_extensions=image_extensions,
)

# Save the full validation report for review before splitting in Notebook 02.
validation_report_path = validation_dir / "validation_report.csv"
validation_df.to_csv(validation_report_path, index=False)

print(f"Saved validation report: {validation_report_path}")
display(validation_df.head(10))


Saved validation report: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\validation\validation_report.csv


,image_name,label_name,base_name,image_path,label_path,status,errors,warnings,image_width,image_height,num_objects,num_M014_Helmet,num_M015_Vest,num_P004_NoThoroughfare,num_W011_Slippery,is_no_sign,has_M014_Helmet,has_M015_Vest,has_P004_NoThoroughfare,has_W011_Slippery
0,M014_001.png,M014_001.txt,M014_001,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1672,941,1,1,0,0,0,False,True,False,False,False
1,M014_002.png,M014_002.txt,M014_002,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1672,941,1,1,0,0,0,False,True,False,False,False
2,M014_003.png,M014_003.txt,M014_003,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1668,943,1,1,0,0,0,False,True,False,False,False
3,M014_004.png,M014_004.txt,M014_004,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1672,941,1,1,0,0,0,False,True,False,False,False
4,M014_005.png,M014_005.txt,M014_005,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1668,943,1,1,0,0,0,False,True,False,False,False
5,M014_006.png,M014_006.txt,M014_006,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1672,941,1,1,0,0,0,False,True,False,False,False
6,M014_007.png,M014_007.txt,M014_007,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1672,941,1,1,0,0,0,False,True,False,False,False
7,M014_008.png,M014_008.txt,M014_008,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1670,942,1,1,0,0,0,False,True,False,False,False
8,M014_009.png,M014_009.txt,M014_009,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1672,941,1,1,0,0,0,False,True,False,False,False
9,M014_010.png,M014_010.txt,M014_010,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1920,1080,1,1,0,0,0,False,True,False,False,False


## 4. Build Pre-Split Profile

The image-level profile is the important handoff to Notebook 02. Its `split_group_key` encodes class presence and no-sign status for stratified train/val/test splitting.


In [4]:
# Build the minimal profile Notebook 02 needs for stratified splitting.
image_profile_df = build_image_level_profile(validation_df)
bbox_records_df = build_bbox_records(validation_df, class_names=class_names)
summary_df = build_input_summary(validation_df)
class_distribution_df = build_class_distribution(validation_df, class_names=class_names)

# Keep Notebook 01 outputs focused: profile CSVs live under reports/profile, not reports/eda.
image_profile_path = profile_dir / "input_image_profile.csv"
bbox_records_path = profile_dir / "input_bbox_records.csv"
summary_path = profile_dir / "input_summary.csv"

image_profile_df.to_csv(image_profile_path, index=False)
bbox_records_df.to_csv(bbox_records_path, index=False)
summary_df.to_csv(summary_path, index=False)

print(f"Saved image profile: {image_profile_path}")
print(f"Saved bbox records: {bbox_records_path}")
print(f"Saved input summary: {summary_path}")


Saved image profile: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\profile\input_image_profile.csv
Saved bbox records: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\profile\input_bbox_records.csv
Saved input summary: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\profile\input_summary.csv


## 5. Review Summary

Use this compact summary to decide whether the dataset is ready for splitting. Invalid rows should be fixed in the source dataset before Notebook 02.


In [5]:
# Compact dataset summary: total valid images, invalid rows, no-sign images, and class counts.
display(summary_df)

# Invalid rows should be fixed in data/input before running Notebook 02.
invalid_rows = validation_df[validation_df["status"] == "invalid"]
print(f"Invalid rows: {len(invalid_rows)}")
if invalid_rows.empty:
    print("No invalid rows found.")
else:
    display(
        invalid_rows[["image_name", "label_name", "base_name", "errors", "warnings"]]
    )

# No-sign images have empty labels and should be preserved for balanced splitting.
no_sign_count = (
    int(image_profile_df["is_no_sign"].sum()) if not image_profile_df.empty else 0
)
print(f"No-sign valid images: {no_sign_count}")


,total_images,valid_images,invalid_rows,no_sign_images,labeled_images,total_objects,num_M014_Helmet,num_M015_Vest,num_P004_NoThoroughfare,num_W011_Slippery
0,475,475,0,30,445,514,129,126,130,129


Invalid rows: 0
No invalid rows found.
No-sign valid images: 30


## 6. Review Class Presence

This table shows both object counts and image counts per sign class before splitting.


In [6]:
# Show object counts and image counts per target sign class.
display(class_distribution_df)

# The plot is optional; if matplotlib is unavailable, the notebook keeps going.
try:
    plot_path = plot_class_distribution(
        class_distribution=class_distribution_df,
        output_path=profile_dir / "input_class_distribution.png",
    )
    print(f"Saved optional class distribution figure: {plot_path}")
except Exception as exc:
    print(f"Skipped class distribution plot: {exc}")


,class_id,class_name,object_count,image_count
0,0,M014_Helmet,129,126
1,1,M015_Vest,126,113
2,2,P004_NoThoroughfare,130,119
3,3,W011_Slippery,129,120


Saved optional class distribution figure: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\profile\input_class_distribution.png


## 7. Notebook 02 Checklist

Before moving to Notebook 02:

- Review and fix any `status = invalid` rows in `validation_report.csv`.
- Confirm the no-sign image count is intentional.
- Confirm each target class appears as expected.
- Confirm `reports/profile/input_image_profile.csv` exists and includes `split_group_key`.
- Use Notebook 02 for train/val/test splitting and full post-split EDA.
